# Обучение моделей предсказывать уровень нарциссизма по текстовым данным

In [ ]:
!pip install -U kaleido

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Библиотеки для ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import precision_recall_fscore_support

# Модели
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Для визуализации
import plotly.graph_objects as go
import json

# Для эмбеддингов
from sentence_transformers import SentenceTransformer

ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ

In [ ]:
# Загружаем итоговый файл, полученный на этапе статистического анализа
data = pd.read_csv('data.csv')
print(f"Исходный датасет загружен: {data.shape}")

# Кодирование целевой переменной
label_encoder = LabelEncoder()
data['narcissism_class_encoded'] = label_encoder.fit_transform(data['narcissism_class'])
print("Классы целевой переменной:", label_encoder.classes_)

In [ ]:
# Функция очистки специальных символов (из оригинального ноутбука)
def clean_special_characters(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    allowed_chars = r'[a-zA-Zа-яА-ЯёЁ0-9\s,.!?;:\-()\"\'«»—]'
    cleaned_chars = [char for char in text if re.match(allowed_chars, char)]
    text = ''.join(cleaned_chars)
    
    patterns_to_replace = [
        (r'\.{3,}', '..'), (r',{3,}', ',,'), (r'!{3,}', '!!'),
        (r'\?{3,}', '??'), (r'-{3,}', '--'), (r':{3,}', '::'), (r';{3,}', ';;')
    ]
    for pattern, replacement in patterns_to_replace:
        text = re.sub(pattern, replacement, text)

    text = re.sub(r'^[,.!?;:\s]+', '', text)
    text = re.sub(r'[,.!?;:\s]+$', '', text)
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    text = re.sub(r'([,!?;:])(?=\S)', r'\1 ', text)
    return text.strip()

In [ ]:
# Функция фильтрации строк с короткими текстовыми блоками
def filter_by_short_blocks(df, text_columns, max_short_blocks=3, min_chars=50):
    df_clean = df.copy()
    for col in text_columns:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].apply(clean_special_characters)
    
    short_counts = []
    for _, row in df_clean.iterrows():
        short_in_row = sum(1 for col in text_columns if len(str(row[col])) < min_chars)
        short_counts.append(short_in_row)
    
    df_clean['short_blocks_count'] = short_counts
    mask = df_clean['short_blocks_count'] <= max_short_blocks
    df_filtered = df_clean[mask].drop(columns=['short_blocks_count']).copy()
    return df_filtered

In [ ]:
# Применяем очистку и фильтрацию
text_features = ['grandiosity', 'fantasy_absorbtion', 'unique_belief', 'attention_seek',
                 'spec_treat', 'manipulation', 'empathy_lack', 'envy', 'arrogance']
df = filter_by_short_blocks(data, text_features)
print(f"Размер после фильтрации коротких ответов: {df.shape}")

TF-IDF ВЕКТОРИЗАЦИЯ И РАЗДЕЛЕНИЕ ДАННЫХ

In [ ]:
# Векторизация каждого текстового блока отдельно
X_vectors = {}
for feature in text_features:
    vectorizer = TfidfVectorizer(max_features=1000)
    X_vectors[feature] = vectorizer.fit_transform(df[feature])

# Объединение всех векторов в одну матрицу признаков
X_combined = np.hstack([X_vectors[feature].toarray() for feature in text_features])
y = df['narcissism_class_encoded'].values
print(f"Размер матрицы признаков (TF-IDF): {X_combined.shape}")

# Разделение на обучающую и тестовую выборки с сохранением баланса классов
df = df.reset_index(drop=True)
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=y
)

X_tfidf_train = X_combined[train_idx]
X_tfidf_test = X_combined[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]
print(f"Обучающая выборка: {X_tfidf_train.shape[0]} наблюдений")
print(f"Тестовая выборка: {X_tfidf_test.shape[0]} наблюдений")

ФУНКЦИЯ ДЛЯ ОБУЧЕНИЯ И ОЦЕНКИ МОДЕЛЕЙ

In [ ]:
def train_and_evaluate_model(model_params, X_train, X_test, y_train, y_test, cv_folds=5,
                             n_iter=10, test_size=0.2, model_name="Model", random_state=42):
    """
    Train model with and cross-validation on training data

    Args:
        model_params: dict with 'model' (estimator) and 'param_dist' (parameter distribution)
        X: features
        y: target
        cv_folds: number of cross-validation folds on training data
        n_iter: number of parameter combinations for random search
        test_size: proportion of data for test set (default 0.2)
        model_name: name of the model
        random_state: random seed for reproducibility

    Returns:
        dict with evaluation results
    """
    from sklearn.preprocessing import LabelEncoder

    # Encode labels if needed
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    class_names = label_encoder.classes_

    print(f"\n{'='*80}")
    print(f"DATA SPLIT FOR {model_name}")
    print(f"{'='*80}")
    print(f"Train size: {len(X_train)} samples ({100*(1-test_size):.0f}%)")
    print(f"Test size:  {len(X_test)} samples ({100*test_size:.0f}%)")
    print(f"Classes: {list(class_names)}")

    # Random search with cross-validation on training data
    search = RandomizedSearchCV(
        estimator=model_params['model'],
        param_distributions=model_params['param_dist'],
        n_iter=n_iter,
        cv=cv_folds,
        scoring='f1_weighted',
        n_jobs=-1,
        random_state=random_state,
        verbose=0
    )

    print(f"\n{'='*80}")
    print(f"RANDOM SEARCH ON TRAINING DATA ({cv_folds}-fold CV)")
    print(f"{'='*80}")
    print(f"Number of iterations: {n_iter}")

    search.fit(X_train, y_train)

    # Display best parameters from random search
    print(f"\nBEST PARAMETERS FOUND:")
    print(f"{search.best_params_}")
    print(f"Best CV score (F1-weighted): {search.best_score_:.4f}")

    # Display random search results summary
    results_df = pd.DataFrame(search.cv_results_)
    print(f"\nTOP 5 PARAMETER COMBINATIONS:")
    print("-" * 80)
    top_indices = results_df['mean_test_score'].nlargest(5).index
    for idx in top_indices:
        print(f"Score: {results_df.loc[idx, 'mean_test_score']:.4f} ± {results_df.loc[idx, 'std_test_score']:.4f}")
        print(f"Params: {results_df.loc[idx, 'params']}")
        print()

    # Train final model on entire training set
    best_model = search.best_estimator_
    best_model.fit(X_train, y_train)

    # Predictions
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    # Calculate metrics on training set
    precision_train, recall_train, f1_train, _ = precision_recall_fscore_support(
        y_train, y_train_pred, average='weighted', zero_division=0
    )

    # Calculate metrics on test set
    precision_test, recall_test, f1_test, _ = precision_recall_fscore_support(
        y_test, y_test_pred, average='weighted', zero_division=0
    )

    # Per-class metrics
    precision_per_class_train, recall_per_class_train, f1_per_class_train, _ = precision_recall_fscore_support(
        y_train, y_train_pred, average=None, zero_division=0
    )

    precision_per_class_test, recall_per_class_test, f1_per_class_test, _ = precision_recall_fscore_support(
        y_test, y_test_pred, average=None, zero_division=0
    )

    # Display final results
    print(f"\n{'='*80}")
    print(f"FINAL MODEL EVALUATION: {model_name}")
    print(f"{'='*80}")

    print(f"\nOVERALL METRICS:")
    print("-" * 50)
    print(f"{'Metric':<15} {'Train':<12} {'Test':<12} {'Difference':<12}")
    print("-" * 50)
    print(f"{'F1-score':<15} {f1_train:.4f}      {f1_test:.4f}      {f1_train - f1_test:.4f}")
    print(f"{'Precision':<15} {precision_train:.4f}      {precision_test:.4f}      {precision_train - precision_test:.4f}")
    print(f"{'Recall':<15} {recall_train:.4f}      {recall_test:.4f}      {recall_train - recall_test:.4f}")

    # Check for overfitting
    f1_diff = f1_train - f1_test
    if f1_diff > 0.05:
        print(f"\nWARNING: Possible overfitting detected (F1 diff = {f1_diff:.4f})")

    print(f"\nMETRICS BY CLASS (TEST SET):")
    print("-" * 70)
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-score':<12}")
    print("-" * 70)

    for i, cls in enumerate(class_names):
        print(f"{str(cls):<20} "
              f"{precision_per_class_test[i]:<12.4f} "
              f"{recall_per_class_test[i]:<12.4f} "
              f"{f1_per_class_test[i]:<12.4f}")

    print("="*80)

    # Return results
    return {
        'model_name': model_name,
        'best_params': search.best_params_,
        'best_cv_score': search.best_score_,
        'train_metrics': {
            'f1': f1_train,
            'precision': precision_train,
            'recall': recall_train,
            'by_class': {
                'precision': precision_per_class_train,
                'recall': recall_per_class_train,
                'f1': f1_per_class_train
            }
        },
        'test_metrics': {
            'f1': f1_test,
            'precision': precision_test,
            'recall': recall_test,
            'by_class': {
                'precision': precision_per_class_test,
                'recall': recall_per_class_test,
                'f1': f1_per_class_test
            }
        },
        'class_names': class_names,
        'best_model': best_model,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }

# ОБУЧЕНИЕ МОДЕЛЕЙ НА TF-IDF ПРИЗНАКАХ

TF-IDF + Логистическая регрессия

In [ ]:
logreg_params = {
    'model': LogisticRegression(random_state=42, max_iter=1000),
    'param_dist': {
        'C': [0.01, 0.1, 1.0, 2.0, 5.0],
        'solver': ['lbfgs', 'liblinear'],
        'class_weight': ['balanced']
    }
}

results = train_and_evaluate_model(
    logreg_params,
    X_tfidf_train,
    X_tfidf_test,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="LogisticRegression"
)

# Access results
print(f"\nBest parameters: {results['best_params']}")
print(f"Test F1-score: {results['test_metrics']['f1']:.4f}")

TF-IDF + Дерево решений

In [ ]:
dt_params = {
    'model': DecisionTreeClassifier(random_state=42),
    'param_dist': {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [3, 5, 10, 15],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [2, 4, 8],
        'class_weight': ['balanced']
    }
}

results_dt = train_and_evaluate_model(
    dt_params,
    X_tfidf_train,
    X_tfidf_test,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="DecisionTreeClassifier"
)

print(f"\nBest parameters: {results_dt['best_params']}")
print(f"Test F1-score: {results_dt['test_metrics']['f1']:.4f}")

TF-IDF + Случайный лес

In [ ]:
rf_params = {
    'model': RandomForestClassifier(random_state=42),
    'param_dist': {
        'n_estimators': [30, 50, 100],
        'max_depth': [5, 8, 15],
        'min_samples_split': [2, 5, 10],
        'class_weight': ['balanced']
    }
}

results_rf = train_and_evaluate_model(
    rf_params,
    X_tfidf_train,
    X_tfidf_test,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="RandomForestClassifier"
)

print(f"\nBest parameters: {results_rf['best_params']}")
print(f"Test F1-score: {results_rf['test_metrics']['f1']:.4f}")

TF-IDF + SVM

In [ ]:
svc_params = {
    'model': SVC(random_state=42),
    'param_dist': {
        'C': [0.1, 1.0, 2.0],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto'],
        'class_weight': ['balanced']
    }
}

results_svc = train_and_evaluate_model(
    svc_params,
    X_tfidf_train,
    X_tfidf_test,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="SVС"
)

print(f"\nBest parameters: {results_svc['best_params']}")
print(f"Test F1-score: {results_svc['test_metrics']['f1']:.4f}")


TF-IDF + XGBoost

In [ ]:
xgb_params = {
    'model': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', class_weight='balanced'),
    'param_dist': {
        'n_estimators': [30, 50, 100],
        'max_depth': [3, 6, 10],
        'learning_rate': [0.01, 0.1, 0.5],
        'reg_alpha': [0.0, 0.1, 0.5],
        'scale_pos_weight': [1, 2, 5]
    }
}

results_xgb = train_and_evaluate_model(
    xgb_params,
    X_tfidf_train,
    X_tfidf_test,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="XGBoost"
)

print(f"\nBest parameters: {results_xgb['best_params']}")
print(f"Test F1-score: {results_xgb['test_metrics']['f1']:.4f}")

TF-IDF + LightGBM

In [ ]:
lgbm_params = {
    'model': LGBMClassifier(random_state=42),
    'param_dist': {
        'n_estimators': [30, 50, 100],
        'max_depth': [3, 6, 10],
        'learning_rate': [0.01, 0.1, 0.5],
        'class_weight': ['balanced']
    }
}

results_lgbm = train_and_evaluate_model(
    lgbm_params,
    X_tfidf_train,
    X_tfidf_test,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="LGBM"
)
 
print(f"\nBest parameters: {results_lgbm['best_params']}")
print(f"Test F1-score: {results_lgbm['test_metrics']['f1']:.4f}")

Визуализация результатов (TF-IDF)

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'XGBoost', 'LightGBM']
df = pd.DataFrame({
    'Model': models,
    'Low': [0.1818, 0.2564, 0.3226, 0.0714, 0.5000, 0.4103],
    'Medium': [0.6024, 0.4688, 0.6744, 0.6517, 0.7073, 0.6133],
    'High': [0.0000, 0.1739, 0.0000, 0.0000, 0.1667, 0.1667],
})

fig = go.Figure()
fig.add_trace(go.Bar(x=df['Model'], y=df['Low'], name='Низкий'))
fig.add_trace(go.Bar(x=df['Model'], y=df['Medium'], name='Средний'))
fig.add_trace(go.Bar(x=df['Model'], y=df['High'], name='Высокий'))

fig.update_layout(
    barmode='group',
    title='F1-score по моделям и классам',
    xaxis_title='Модель',
    yaxis_title='F1-score',
    legend_title_text='Класс'
)

fig.update_yaxes(range=[0, 0.8])
fig.update_traces(cliponaxis=False)

ЭМБЕДДИНГИ НА ОСНОВЕ LLM + БАЗОВЫЕ МОДЕЛИ

In [ ]:
# Подготовка данных для эмбеддингов
QUESTIONS = {
    1: "Представь, что команда работает над проектом, но принимает решение, которое ты считаешь неэффективным...",
    # (полный текст вопросов опущен для краткости, должен быть как в оригинале)
    2: "Расскажи о случае, когда ты мечтал(а) о будущем успехе...",
    3: "Представь, что ты делишься своей идеей с группой, а тебе говорят, что она странная или непрактичная...",
    4: "Представь, что ты сделал(а) вклад в проект, но никто особо не обратил на это внимания...",
    5: "Представь, что преподаватель/начальник делает тебе замечание...",
    6: "Представь, что, чтобы переубедить кого-то, тебе нужно лишь скрыть информацию...",
    7: "Представь, что кто-то из группы расстроен и делится своими переживаниями...",
    8: "Представим, что в вашей группе вы делаете проект в разных командах...",
    9: "Ты сталкиваешься с тем, что кто-то из группы постоянно ошибается..."
}

# Переименовываем колонки для удобства
ans_cols_map = {
    'grandiosity': 'ans_1', 'fantasy_absorbtion': 'ans_2',
    'unique_belief': 'ans_3', 'attention_seek': 'ans_4',
    'spec_treat': 'ans_5', 'manipulation': 'ans_6',
    'empathy_lack': 'ans_7', 'envy': 'ans_8', 'arrogance': 'ans_9'
}
data_emb = df[list(ans_cols_map.keys()) + ['narcissism_class']].copy()
data_emb.rename(columns=ans_cols_map, inplace=True)

# Создаем объединенный текст (Вопрос + Ответ)
for n in range(1, 10):
    col_name = f'text_{n}'
    question = QUESTIONS[n].replace('\n', ' ')
    data_emb[col_name] = data_emb[f'ans_{n}'].apply(
        lambda answer: f"Вопрос: {question}\nОтвет: {answer}" if pd.notna(answer) else f"Вопрос: {question}\nОтвет: "
    )
text_cols = [f'text_{i}' for i in range(1, 10)]
data_emb['text'] = data_emb[text_cols].fillna('').astype(str).agg(' '.join, axis=1)

In [ ]:
# Функция для обрезки длинных текстов
def smart_truncate(text, max_chars=12000, overlap=2000):
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        start = end - overlap
    return chunks[:4]

# Загрузка модели и создание эмбеддингов
print("Загрузка модели SentenceTransformer...")
model = SentenceTransformer('BAAI/bge-m3')

print("Создание эмбеддингов для текстов (вопрос + ответ)...")
data_emb['text_chunks'] = data_emb['text'].apply(smart_truncate)
texts = [' '.join(chunks) for chunks in data_emb['text_chunks']]
embeddings = model.encode(texts, normalize_embeddings=True, batch_size=32)
data_emb['text_emb'] = list(embeddings)

In [ ]:
X_text = np.vstack(data_emb['text_emb'].values)
X_train_text = X_text[train_idx]
X_test_text = X_text[test_idx]
print(f"Размер матрицы эмбеддингов: {X_text.shape}")

Эмбеддинги + Логистическая регрессия (с тем же перебором гиперпараметров)

In [ ]:
results_lg_text = train_and_evaluate_model(
    logreg_params,
    X_train_text,
    X_test_text,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="LogisticRegression"
)

print(f"\nBest parameters: {results_lg_text['best_params']}")
print(f"Test F1-score: {results_lg_text['test_metrics']['f1']:.4f}")

Эмбеддинги + деревья решений

In [ ]:
results_dt_text = train_and_evaluate_model(
    dt_params,
    X_train_text,
    X_test_text,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="DecisionTreeClassifier"
)

print(f"\nBest parameters: {results_dt_text['best_params']}")
print(f"Test F1-score: {results_dt_text['test_metrics']['f1']:.4f}")

Эмбеддинги + Случайный лес

In [ ]:
results_rf_text = train_and_evaluate_model(
    rf_params,
    X_train_text,
    X_test_text,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="RandomForestClassifier"
)
 
print(f"\nBest parameters: {results_rf_text['best_params']}")
print(f"Test F1-score: {results_rf_text['test_metrics']['f1']:.4f}")

Эмбеддинги + SVM

In [ ]:
results_svc_text = train_and_evaluate_model(
    svc_params,
    X_train_text,
    X_test_text,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="SVС"
)
 
print(f"\nBest parameters: {results_svc_text['best_params']}")
print(f"Test F1-score: {results_svc_text['test_metrics']['f1']:.4f}")

Эмбеддинги + XGBoost

In [ ]:
results_xgb_text = train_and_evaluate_model(
    xgb_params,
    X_train_text,
    X_test_text,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="XGBoost"
)
 
print(f"\nBest parameters: {results_xgb_text['best_params']}")
print(f"Test F1-score: {results_xgb_text['test_metrics']['f1']:.4f}")

Эмбеддинги + LightGBM

In [ ]:
results_lgbm_text = train_and_evaluate_model(
    lgbm_params,
    X_train_text,
    X_test_text,
    y_train,
    y_test,
    cv_folds=5,
    n_iter=10,
    test_size=0.2,
    model_name="LGBM"
)
 
print(f"\nBest parameters: {results_lgbm_text['best_params']}")
print(f"Test F1-score: {results_lgbm_text['test_metrics']['f1']:.4f}")

Визуализация результатов (Embeddings)

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'XGBoost', 'LightGBM']
df = pd.DataFrame({
    'Model': models,
    'Low': [0.3429, 0.2449, 0.1739, 0.1538, 0.1481, 0.1212],
    'Medium': [0.6000, 0.3448, 0.7021, 0.5672, 0.6742, 0.5783],
    'High': [0.3810, 0.2105, 0.0000, 0.3030, 0.0000, 0.0000],
})

fig = go.Figure()
fig.add_trace(go.Bar(x=df['Model'], y=df['Low'], name='Низкий'))
fig.add_trace(go.Bar(x=df['Model'], y=df['Medium'], name='Средний'))
fig.add_trace(go.Bar(x=df['Model'], y=df['High'], name='Высокий'))

fig.update_layout(
    barmode='group',
    title='F1-score по моделям и классам (эмбеддинги)',
    xaxis_title='Модель',
    yaxis_title='F1-score',
    legend_title_text='Класс'
)

fig.update_yaxes(range=[0, 0.8])
fig.update_traces(cliponaxis=False)

СРАВНЕНИЕ ПОДХОДОВ ПО ОБЩЕМУ F1-score: TF-IDF vs Embeddings

In [ ]:
models = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'XGBoost', 'LightGBM']

df = pd.DataFrame({
    'Model': models,
    'TF-IDF': [0.1818, 0.2564, 0.3226, 0.0714, 0.5, 0.4103],
    'Embeddings': [0.4912, 0.2955, 0.4425, 0.4048, 0.4192, 0.3578]
})

fig = go.Figure()
fig.add_trace(go.Bar(x=df['Model'], y=df['TF-IDF'], name='TF-IDF'))
fig.add_trace(go.Bar(x=df['Model'], y=df['Embeddings'], name='Embeddings'))

fig.update_layout(
    barmode='group',
    title={
        'text': "Общий F1-score у моделей TF-IDF vs embeddings<br><span style='font-size: 18px; font-weight: normal;'></span>"
    },
    xaxis_title='Model',
    yaxis_title='F1-score',
    legend_title_text='Vectorizer'
)

fig.update_xaxes(tickangle=45)
fig.update_yaxes(range=[0, 1])
fig.update_traces(cliponaxis=False)

Выводы:
1. Для TF-IDF лучший результат показал XGBoost (0.5676), но с сильным переобучением и плохим качеством предсказания по миноритарным классам.
2. Для эмбеддингов лучший результат у Логистической регрессии (0.4912) — более стабильная модель.
3. Класс 'High' (высокий нарциссизм) плохо классифицируется всеми моделями из-за малого количества примеров.
4. Эмбеддинги дают более сбалансированные результаты между классами.